Content based filtering

In [4]:
import pandas as pd

movies=pd.read_csv("../datasets/movies.csv")
movies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [5]:
ratings=pd.read_csv("../datasets/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [7]:
ratings.duplicated().sum()

np.int64(0)

In [8]:
ratings['movieId'].unique()

array([     1,      3,      6, ..., 160836, 163937, 163981], shape=(9724,))

In [9]:
ratings['rating'].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [10]:
movies.duplicated().sum()

np.int64(0)

In [11]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [12]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='str')

In [13]:
with pd.option_context('display.max_rows',None):
    print(movies['genres'].value_counts())

genres
Drama                                                                            1053
Comedy                                                                            946
Comedy|Drama                                                                      435
Comedy|Romance                                                                    363
Drama|Romance                                                                     349
Documentary                                                                       339
Comedy|Drama|Romance                                                              276
Drama|Thriller                                                                    168
Horror                                                                            167
Horror|Thriller                                                                   135
Crime|Drama                                                                       134
Crime|Drama|Thriller                           

In [14]:
movies['genres']=movies['genres'].replace('(no genres listed)','')

In [15]:
movie_ratings=pd.merge(movies,ratings,on='movieId',how='outer')
movie_ratings.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,4.0,9.649827e+08
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0,8.474350e+08
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7.0,4.5,1.106636e+09
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15.0,2.5,1.510578e+09
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17.0,4.5,1.305696e+09


In [16]:
movie_ratings=pd.merge(movies,ratings,on='movieId',how='inner')
movie_ratings.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483


In [17]:
movie_ratings=pd.merge(ratings,movies,on='movieId',how='outer')
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1.0,1,4.0,9.649827e+08,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,5.0,1,4.0,8.474350e+08,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,7.0,1,4.5,1.106636e+09,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,15.0,1,2.5,1.510578e+09,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,17.0,1,4.5,1.305696e+09,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


In [18]:
movie_ratings=pd.merge(ratings,movies,on='movieId',how='inner')
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [19]:
movies['genre_text']=movies['genres'].str.replace('|',' ',regex=False)
movies[['title','genres','genre_text']].head()

,title,genres,genre_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer()
tfidf_matrix=tfidf.fit_transform(movies['genre_text'])
print("TF-IDF Matrix Shape:",tfidf_matrix.shape)

TF-IDF Matrix Shape: (9742, 21)


In [21]:
feature_names=tfidf.get_feature_names_out()
feature_names

array(['action', 'adventure', 'animation', 'children', 'comedy', 'crime',
       'documentary', 'drama', 'fantasy', 'fi', 'film', 'horror', 'imax',
       'musical', 'mystery', 'noir', 'romance', 'sci', 'thriller', 'war',
       'western'], dtype=object)

In [22]:
print(tfidf_matrix[0].toarray()) 

[[0.         0.41684567 0.51622547 0.50484547 0.26758648 0.
  0.         0.         0.48299014 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]]


User Preference

In [ ]:
user_id=1
user_data=movie_ratings[movie_ratings['userId']==user_id]
print("User ID: ",user_id)
print("Total Movie rated: ",len(user_data))

User ID:  1
Total Movie rated:  232


In [24]:
liked_movies=user_data[user_data['rating']>=4].copy()
print("Number of liked movies:", len(liked_movies))

Number of liked movies: 200


In [25]:
movie_indices=pd.Series(movies.index,index=movies['movieId'])
liked_indices=movie_indices.loc[liked_movies['movieId']].values
liked_vectors=tfidf_matrix[liked_indices]
weights=liked_movies['rating'].values
print("Liked movies vectors:",liked_vectors)

Liked movies vectors: <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 629 stored elements and shape (200, 21)>
  Coords	Values
  (0, 1)	0.41684567364693936
  (0, 2)	0.5162254711770092
  (0, 3)	0.5048454681396087
  (0, 4)	0.26758647689140014
  (0, 8)	0.482990142708577
  (1, 4)	0.5709154064399099
  (1, 16)	0.8210088907493954
  (2, 0)	0.5493281743985543
  (2, 5)	0.6359470441562757
  (2, 18)	0.5420423542868653
  (3, 18)	0.5669745241208175
  (3, 14)	0.8237353270304562
  (4, 5)	0.5538536336398807
  (4, 18)	0.47207095349706946
  (4, 14)	0.6858536049101646
  (5, 1)	0.5505900204002118
  (5, 4)	0.35344122077957074
  (5, 16)	0.5082686179846321
  (5, 5)	0.5599937007337968
  (6, 7)	0.33878449184405596
  (6, 0)	0.5020442850786316
  (6, 19)	0.7957239495615616
  (7, 16)	0.4665822429924612
  (7, 7)	0.29964736811657006
  (7, 0)	0.4440470338029034
  :	:
  (192, 9)	0.4908715429647324
  (193, 7)	0.3475796486674422
  (193, 0)	0.5150778162049144
  (193, 5)	0.5962960394023562
  (193, 18)	0.50824

In [26]:
print("Ratings",weights)

Ratings [4. 4. 4. 5. 5. 5. 4. 5. 5. 5. 5. 5. 4. 5. 5. 4. 4. 5. 4. 4. 5. 4. 5. 4.
 4. 5. 4. 4. 4. 5. 5. 5. 4. 4. 5. 5. 5. 4. 5. 5. 5. 5. 5. 5. 5. 4. 5. 4.
 5. 5. 5. 4. 5. 5. 4. 5. 5. 5. 5. 5. 4. 5. 5. 4. 5. 5. 5. 5. 5. 5. 4. 5.
 5. 5. 5. 5. 5. 4. 4. 4. 5. 4. 5. 5. 5. 4. 5. 5. 4. 4. 5. 4. 4. 5. 5. 4.
 4. 5. 4. 5. 4. 5. 4. 5. 4. 5. 5. 5. 5. 4. 4. 4. 5. 5. 5. 5. 5. 4. 5. 4.
 4. 4. 4. 5. 5. 5. 4. 5. 5. 4. 5. 4. 5. 5. 4. 4. 5. 5. 5. 5. 5. 5. 4. 4.
 4. 5. 4. 4. 5. 5. 5. 5. 5. 4. 4. 5. 5. 5. 4. 5. 5. 5. 5. 5. 4. 5. 5. 5.
 4. 5. 5. 5. 5. 4. 5. 4. 5. 5. 5. 4. 4. 5. 5. 5. 4. 4. 4. 5. 4. 4. 5. 5.
 5. 5. 4. 4. 5. 4. 4. 5.]


In [28]:
import numpy as np

user_profile=np.average(liked_vectors.toarray(),axis=0,weights=weights)
print(user_profile)

print(feature_names)

[0.18721029 0.19791362 0.07553719 0.10188425 0.16948331 0.12309958
 0.         0.13065138 0.12044862 0.08269059 0.00314613 0.0303705
 0.         0.06694019 0.04557201 0.00314613 0.06878199 0.08269059
 0.10835758 0.0808083  0.02431303]
['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'horror' 'imax' 'musical'
 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war' 'western']


user 1

In [30]:
profile_df=pd.DataFrame({'genre':feature_names,'preference_score':user_profile})

profile_df=profile_df.sort_values(by='preference_score',ascending=False)
profile_df

,genre,preference_score
1,adventure,0.197914
0,action,0.187210
4,comedy,0.169483
7,drama,0.130651
5,crime,0.123100
8,fantasy,0.120449
18,thriller,0.108358
3,children,0.101884
17,sci,0.082691
9,fi,0.082691


Similarity Score - user 1 profile and tfidf matrix-entire movie

In [32]:
similarity_scores=cosine_similarity(user_profile.reshape(1,-1),tfidf_matrix).flatten()
print("Similarity Scores")
print(similarity_scores[:10])
print("Total Scores:", len(similarity_scores))

Similarity Scores
[0.60251981 0.51463256 0.33395815 0.42821706 0.3693785  0.52275822
 0.33395815 0.44586013 0.40801337 0.62607522]
Total Scores: 9742


In [50]:
movies['similarity_score']=similarity_scores
movies.head()

,movieId,title,genres,genre_text,similarity_scores,similarity_score
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy,0.602520,0.602520
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,0.514633,0.514633
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,0.333958,0.333958
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,0.428217,0.428217
4,5,Father of the Bride Part II (1995),Comedy,Comedy,0.369379,0.369379


In [51]:
watched_movie_ids=user_data['movieId'].unique()
unwatched_movies=movies[~movies['movieId'].isin(watched_movie_ids)].copy()
print("Movies available for recommendation: ",len(unwatched_movies))

Movies available for recommendation:  9510


In [52]:
unwatched_movies.head()

,movieId,title,genres,genre_text,similarity_scores,similarity_score
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,0.514633,0.514633
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,0.428217,0.428217
4,5,Father of the Bride Part II (1995),Comedy,Comedy,0.369379,0.369379
6,7,Sabrina (1995),Comedy|Romance,Comedy Romance,0.333958,0.333958
7,8,Tom and Huck (1995),Adventure|Children,Adventure Children,0.445860,0.445860


In [54]:
recommendations=unwatched_movies.sort_values(by='similarity_score',ascending=False)
recommendations.head()

,movieId,title,genres,genre_text,similarity_scores,similarity_score
8597,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,Action Adventure Comedy Drama Fantasy Thriller,0.792251,0.792251
4681,6990,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,Action Adventure Comedy Crime Drama,0.777282,0.777282
4005,5657,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,Action Adventure Comedy Crime Drama,0.777282,0.777282
6570,55116,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,Action Adventure Comedy Drama Thriller,0.772458,0.772458
3608,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,Action Adventure Comedy Drama Romance Thriller,0.756381,0.756381


Function for multiple user recommendation

In [75]:
user_id=int(input("Enter the user id: "))


def recommend_movie(user_id):
    user_data=movie_ratings[movie_ratings['userId']==user_id]
    print("User ID: ",user_id)
    print("Total Movie rated: ",len(user_data))
    
    liked_movies=user_data[user_data['rating']>=4].copy()
    print("Number of liked movies:", len(liked_movies))
    
    watched_movie_ids=user_data['movieId'].unique()
    unwatched_movies=movies[~movies['movieId'].isin(watched_movie_ids)].copy()
    print("Movies available for recommendation: ",len(unwatched_movies))
    
    recommendations=unwatched_movies.sort_values(by='similarity_score',ascending=False)
    
    return (recommendations.head())
    
recommend_movie(user_id)    

User ID:  3
Total Movie rated:  39
Number of liked movies: 16
Movies available for recommendation:  9703


,movieId,title,genres,genre_text,similarity_scores,similarity_score
8597,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,Action Adventure Comedy Drama Fantasy Thriller,0.792251,0.792251
4681,6990,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,Action Adventure Comedy Crime Drama,0.777282,0.777282
4005,5657,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,Action Adventure Comedy Crime Drama,0.777282,0.777282
6570,55116,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,Action Adventure Comedy Drama Thriller,0.772458,0.772458
3608,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,Action Adventure Comedy Drama Romance Thriller,0.756381,0.756381
